# Bagheria 2026 — thread Genere

Gap di genere fra i giovani a Bagheria, confrontato con Palermo, Sicilia e Italia.

**Prerequisiti**: `pipeline.fetch`, `pipeline.build` e `notebooks/analisi.ipynb` già eseguiti.
Le definizioni delle misure (tasso di occupazione, disoccupazione, fuori da lavoro e studio)
non vengono ridefinite qui: si riusano quelle del notebook condiviso, leggendo
`data/processed/analisi_condizione_15_24.csv`. Serve a garantire che i numeri di questo thread
e quelli degli altri due si sommino nella stessa proposal.

**Fasce d'età**: 15-24 sul lavoro, 9-24 sull'istruzione. Non sono scelte, sono le uniche
classi giovanili disponibili a livello comunale — la cella di verifica qui sotto lo mostra.

# **Caricamento**

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

RADICE = Path.cwd()
if RADICE.name == "notebooks":
    RADICE = RADICE.parent
PROCESSED = RADICE / "data" / "processed"


def leggi(nome: str) -> pd.DataFrame:
    """Codici come stringhe: `condizione` contiene 1, 12, 99 e pandas li leggerebbe come interi."""
    tabella = pd.read_csv(PROCESSED / nome, dtype=str)
    for colonna in ("valore", "eta_anni", "popolazione", "occupati", "tasso_occupazione",
                    "tasso_disoccupazione", "fuori_da_lavoro_e_studio"):
        if colonna in tabella.columns:
            tabella[colonna] = pd.to_numeric(tabella[colonna])
    if "anno" in tabella.columns:
        tabella["anno"] = tabella["anno"].astype(int)
    return tabella


atteso = PROCESSED / "analisi_condizione_15_24.csv"
if not atteso.exists():
    raise SystemExit(f"manca {atteso.name}: esegui prima notebooks/analisi.ipynb")

istr_lav = leggi("censpop_istr_lav_long.csv")
ottomila = leggi("ottomilacensus_long.csv")
indicatori = leggi("indicatori.csv")
codici = leggi("codici.csv")
territori = leggi("territori.csv")
condizione = leggi("analisi_condizione_15_24.csv")
popolazione = leggi("censpop_popolazione_long.csv")

BAGHERIA, PALERMO, SICILIA, ITALIA = "082006", "082053", "ITG1", "IT"
CONFRONTO = [BAGHERIA, PALERMO, SICILIA, ITALIA]
NOMI = territori.set_index("territorio")["nome_territorio"].to_dict()
ORDINE = ["Bagheria", "Palermo", "Sicilia", "Italia"]

etichette = codici.set_index(["dimensione", "codice"])["etichetta"].to_dict()
print("misure disponibili dal notebook condiviso:", [c for c in condizione.columns if "tasso" in c or "fuori" in c])

misure disponibili dal notebook condiviso: ['tasso_occupazione', 'tasso_disoccupazione', 'fuori_da_lavoro_e_studio']


# **Verifica di fattibilità del thread**
Il piano iniziale del thread prevedeva la decomposizione del gap occupazionale per titolo di
studio. Prima di scrivere l'analisi, si controlla che l'incrocio esista.

In [2]:
lavoro = istr_lav[istr_lav["tavola"].eq("lavoro")]
istruzione = istr_lav[istr_lav["tavola"].eq("istruzione")]

print("Tavola LAVORO — titoli di studio presenti:", sorted(lavoro["titolo_studio"].unique()))
print("Tavola ISTRUZIONE — titoli di studio presenti:", sorted(istruzione["titolo_studio"].unique()))
print("Tavola ISTRUZIONE — condizioni professionali presenti:", sorted(istruzione["condizione"].unique()))
print()
print("Classi d'età per tavola:")
print("  lavoro:    ", sorted(lavoro["eta"].unique()))
print("  istruzione:", sorted(istruzione["eta"].unique()))
print()
print("Anni con dato sulla classe 15-24 (lavoro):", sorted(lavoro[lavoro["eta"].eq("Y15-24")]["anno"].unique().tolist()))
print("Anni con dato sulla classe 9-24 (istruzione):", sorted(istruzione[istruzione["eta"].eq("Y9-24")]["anno"].unique().tolist()))

Tavola LAVORO — titoli di studio presenti: ['ALL']
Tavola ISTRUZIONE — titoli di studio presenti: ['ALL', 'BL', 'IL', 'LBNA', 'LSE', 'ML', 'ML_RDD', 'NED', 'PSE', 'RDD', 'USE_IF']
Tavola ISTRUZIONE — condizioni professionali presenti: ['99']

Classi d'età per tavola:
  lavoro:     ['Y15-24', 'Y25-49', 'Y50-64', 'Y_GE15', 'Y_GE65']
  istruzione: ['Y25-49', 'Y50-64', 'Y9-24', 'Y_GE65', 'Y_GE9']

Anni con dato sulla classe 15-24 (lavoro): [2018, 2019, 2021, 2022, 2023, 2024]
Anni con dato sulla classe 9-24 (istruzione): [2018, 2019, 2020, 2021, 2022, 2023, 2024]


**📌 Risultato chiave** — L'incrocio *titolo di studio × condizione professionale* non esiste a livello comunale: nella tavola lavoro il titolo è solo `ALL`, in quella istruzione la condizione è solo `99`. La decomposizione del gap per titolo di studio non è difficile, è impossibile con questi dati — il piano del thread cambia di conseguenza.

> **La decomposizione per titolo di studio non è possibile.** Nella tavola sul lavoro il titolo
> di studio esiste solo come `ALL`: il censimento permanente non pubblica a livello comunale
> l'incrocio condizione professionale × titolo di studio. Nella tavola sull'istruzione vale il
> simmetrico: i titoli di studio ci sono tutti, ma la condizione professionale è solo `99`, il totale.
>
> Le due tavole si toccano solo sui totali, quindi *"tra le diplomate, quante lavorano"* non è una
> domanda a cui questi dati rispondono. Non è aggirabile con un'aggregazione diversa.
>
> **Cosa si fa invece.** Si misurano i due gap separatamente — occupazione (15-24) e istruzione
> (9-24) — e si guarda se vanno nella stessa direzione. È una domanda diversa e più debole della
> decomposizione, ma è onesta: dice se le ragazze di Bagheria sono svantaggiate *anche* nello
> studio o *solo* nel lavoro.
>
> Nota sulle fasce: lavoro 15-24, istruzione 9-24. Non sono sovrapponibili e non vanno mai
> mostrate come se fossero la stessa popolazione.

# **Gap di genere sull'occupazione, 15-24**
Le tre misure definite nel notebook condiviso, lette per genere. Il 2020 manca alla fonte.

In [3]:
giovani = condizione[condizione["territorio"].isin(CONFRONTO)].copy()

# Serie di Bagheria: le tre misure per maschi e femmine.
bagheria = (giovani[giovani["territorio"].eq(BAGHERIA) & giovani["genere"].isin(["M", "F"])]
    .pivot_table(index="anno", columns="genere",
                 values=["tasso_occupazione", "tasso_disoccupazione", "fuori_da_lavoro_e_studio"]))
bagheria.round(1)

fuori_da_lavoro_e_studio       tasso_disoccupazione       tasso_occupazione      
genere                        F     M                    F     M                 F     M
anno                                                                                    
2018                       38.0  37.4                 79.0  61.7               4.7  11.5
2019                       32.7  32.8                 76.8  61.9               5.2  11.8
2021                       29.9  29.7                 59.2  47.0               6.2  13.8
2022                       27.8  26.8                 54.3  41.8               7.7  15.4
2023                       31.1  30.4                 56.2  45.2               8.0  15.2
2024                       26.7  27.1                 45.6  35.1               8.2  16.5

In [4]:
# Il gap in punti percentuali (maschi meno femmine) sulle tre misure, per i quattro territori.
def gap_di_genere(misura: str) -> pd.DataFrame:
    largo = (giovani[giovani["genere"].isin(["M", "F"])]
             .pivot_table(index=["nome_territorio", "anno"], columns="genere", values=misura))
    return (largo["M"] - largo["F"]).rename(misura)


gap = pd.concat([gap_di_genere(m) for m in
                 ["tasso_occupazione", "tasso_disoccupazione", "fuori_da_lavoro_e_studio"]], axis=1)
gap = gap.round(1).reset_index()
gap["nome_territorio"] = pd.Categorical(gap["nome_territorio"], ORDINE, ordered=True)
gap = gap.sort_values(["nome_territorio", "anno"])
gap.to_csv(PROCESSED / "genere_gap_occupazione.csv", index=False)

gap.pivot(index="anno", columns="nome_territorio", values="tasso_occupazione")

nome_territorio,Bagheria,Palermo,Sicilia,Italia
anno,,,,
2018,6.8,5.7,7.3,8.2
2019,6.6,5.2,7.4,8.5
2021,7.6,6.4,8.8,9.8
2022,7.7,6.7,9.3,9.7
2023,7.2,6.3,9.4,9.5
2024,8.3,6.9,9.9,9.6


**📌 Risultato chiave** — A Bagheria fra 2018 e 2024 il tasso di occupazione femminile 15-24 sale da 4.7% a 8.2% e quello maschile da 11.5% a 16.5%: entrambi crescono, ma il gap in punti non si chiude, si allarga (6.8 → 8.3 pp).

> Un gap positivo sul tasso di occupazione significa che gli uomini lavorano di più. Sul
> `fuori_da_lavoro_e_studio` un gap positivo significa il contrario di quel che sembra: sono
> *gli uomini* a essere più spesso fuori da lavoro e studio. Le due misure vanno lette insieme,
> perché una fascia dove molti studiano ancora nasconde metà del fenomeno.

# **Quanto è preciso il gap? Intervalli di confidenza**
Prima di confrontare i gap fra territori serve l'ordine di grandezza dell'errore di ogni
numero: Bagheria è un comune, e i suoi conteggi sono piccoli. CI di Wilson sui tassi,
CI di Newcombe sulla differenza M-F.

Per gap occupazionale (o divario di genere nell'occupazione) si intende la differenza 
tra il tasso di occupazione maschile e quello femminile all'interno 
di una determinata popolazione o fascia d'età.

Avvertenza di metodo: i conteggi comunali non sono interi perché il censimento permanente
è una stima da registro + campione. I CI binomiali qui sotto trattano
i conteggi come esatti e sono quindi un **limite inferiore** dell'incertezza vera.

In [5]:
from statsmodels.stats.proportion import proportion_confint


def wilson(successi, totale):
    return proportion_confint(successi, totale, alpha=0.05, method="wilson")


def gap_con_ci(occ_m, pop_m, occ_f, pop_f):
    """Gap M-F fra proporzioni con CI 95% di Newcombe, costruito sui limiti di Wilson."""
    p_m, p_f = occ_m / pop_m, occ_f / pop_f
    l_m, u_m = wilson(occ_m, pop_m)
    l_f, u_f = wilson(occ_f, pop_f)
    g = p_m - p_f
    return (g,
            g - ((p_m - l_m) ** 2 + (u_f - p_f) ** 2) ** 0.5,
            g + ((u_m - p_m) ** 2 + (p_f - l_f) ** 2) ** 0.5)


# Conteggi non arrotondati dal notebook condiviso: per questo qualche decimale può
# differire dalla tabella dei gap sopra, che parte dai tassi già arrotondati.
conteggi_occ = (giovani[giovani["genere"].isin(["M", "F"])]
                .pivot_table(index=["territorio", "nome_territorio", "anno"],
                             columns="genere", values=["occupati", "popolazione"]))
conteggi_occ.columns = [f"{misura}_{gen}" for misura, gen in conteggi_occ.columns]

righe = []
for (territorio, nome, anno), r in conteggi_occ.iterrows():
    g, lo, hi = gap_con_ci(r["occupati_M"], r["popolazione_M"], r["occupati_F"], r["popolazione_F"])
    righe.append({"territorio": territorio, "nome_territorio": nome, "anno": anno,
                  "tasso_F": 100 * r["occupati_F"] / r["popolazione_F"],
                  "tasso_M": 100 * r["occupati_M"] / r["popolazione_M"],
                  "gap": 100 * g, "gap_lo": 100 * lo, "gap_hi": 100 * hi})
ci_gap = pd.DataFrame(righe)
ci_gap["rapporto_M_F"] = ci_gap["tasso_M"] / ci_gap["tasso_F"]
ci_gap = ci_gap.round({"tasso_F": 1, "tasso_M": 1, "gap": 1, "gap_lo": 1, "gap_hi": 1, "rapporto_M_F": 2})
ci_gap.to_csv(PROCESSED / "genere_gap_occupazione_ci.csv", index=False)

print("Bagheria — gap occupazionale M-F (punti) con CI 95%:")
print(ci_gap[ci_gap["territorio"].eq(BAGHERIA)][["anno", "tasso_F", "tasso_M", "gap", "gap_lo", "gap_hi"]]
      .to_string(index=False))
print("\nAnno 2024, i quattro territori:")
ci_gap[ci_gap["anno"].eq(2024)].set_index("nome_territorio").reindex(ORDINE)[
    ["tasso_F", "tasso_M", "gap", "gap_lo", "gap_hi"]]

Bagheria — gap occupazionale M-F (punti) con CI 95%:
 anno  tasso_F  tasso_M  gap  gap_lo  gap_hi
 2018      4.7     11.5  6.9     5.5     8.2
 2019      5.2     11.8  6.7     5.3     8.1
 2021      6.2     13.8  7.6     6.1     9.1
 2022      7.7     15.4  7.7     6.1     9.3
 2023      8.0     15.2  7.2     5.5     8.8
 2024      8.2     16.5  8.3     6.6    10.0

Anno 2024, i quattro territori:


,tasso_F,tasso_M,gap,gap_lo,gap_hi
nome_territorio,,,,,
Bagheria,8.2,16.5,8.3,6.6,10.0
Palermo,9.6,16.5,6.9,6.4,7.5
Sicilia,10.4,20.3,9.9,9.7,10.1
Italia,17.3,26.9,9.7,9.6,9.7


**📌 Risultato chiave** — Gap occupazionale 2024 di Bagheria: **8.3 pp** [CI 95% 6.6-10.0]. Il gap è solido — nessun CI tocca lo zero, in nessun anno e in nessun territorio — ma la precisione comunale è di ±1.5-1.7 pp: le oscillazioni annue (7.7 → 7.2 → 8.3) sono rumore, non trend.

> Il gap esiste ed è solido: in nessun anno e in nessun territorio il CI 95% tocca lo zero.
> Ma la precisione comunale è di ±1.5-1.7 punti: le oscillazioni anno su anno di Bagheria
> (7.7 → 7.2 → 8.3 fra 2022 e 2024) stanno tutte dentro gli intervalli e non vanno
> raccontate come peggioramenti o recuperi annuali. Il confronto sensato è fra territori
> e su più anni, ed è quello che fanno le sezioni successive.

# **Punti percentuali o rapporto? Entrambi**
Il gap in punti risponde a "quanti punti separano i tassi"; il rapporto M/F a "quante
volte è più probabile che un ragazzo lavori rispetto a una coetanea". Con tassi base
molto diversi fra territori le due scale possono ordinare i territori in modo opposto:
riportarne una sola sarebbe una scelta di comodo.

In [6]:
# Il gap in punti e il rapporto fra i tassi, fianco a fianco.
print("Rapporto M/F sul tasso di occupazione 15-24, serie:")
print(ci_gap.pivot(index="anno", columns="nome_territorio", values="rapporto_M_F")[ORDINE].to_string())
print("\nAnno 2024, le due scale:")
ci_gap[ci_gap["anno"].eq(2024)].set_index("nome_territorio").reindex(ORDINE)[
    ["tasso_F", "tasso_M", "gap", "rapporto_M_F"]]

Rapporto M/F sul tasso di occupazione 15-24, serie:
nome_territorio  Bagheria  Palermo  Sicilia  Italia
anno                                               
2018                 2.47     1.90     1.97    1.58
2019                 2.29     1.75     1.93    1.59
2021                 2.23     1.84     2.04    1.65
2022                 2.00     1.77     2.00    1.59
2023                 1.89     1.68     1.94    1.56
2024                 2.01     1.72     1.95    1.56

Anno 2024, le due scale:


,tasso_F,tasso_M,gap,rapporto_M_F
nome_territorio,,,,
Bagheria,8.2,16.5,8.3,2.01
Palermo,9.6,16.5,6.9,1.72
Sicilia,10.4,20.3,9.9,1.95
Italia,17.3,26.9,9.7,1.56


**📌 Risultato chiave** — Le due scale ordinano i territori in modo opposto. **In punti** Bagheria (8.3) sta sopra Palermo (6.9) ma sotto Sicilia (9.9) e Italia (9.7); **in rapporto M/F** (2.01) è la peggiore del panel (Palermo 1.72, Sicilia 1.95, Italia 1.56). Il tratto locale non è l'ampiezza del divario: è il **livello** del tasso femminile, 8.2%, il più basso dei quattro territori.

> Le due scale raccontano storie opposte, ed è il motivo per cui vanno dichiarate entrambe:
>
> - **in punti**, il gap 2024 di Bagheria (8.3) è sopra Palermo (6.9) ma sotto Sicilia (9.9)
>   e Italia (9.7);
> - **in rapporto**, Bagheria è la peggiore delle quattro: un ragazzo ha il doppio della
>   probabilità di lavorare di una coetanea (2.0, contro l'1.6 nazionale).
>
> Quando i tassi sono bassi per entrambi i generi, gli stessi punti percentuali pesano
> molto di più. Le scale divergono anche nel tempo: dal 2018 il gap in punti sale
> (6.9 → 8.3) mentre il rapporto scende (2.47 → 2.01), perché entrambi i tassi crescono.
> Ogni claim della proposal deve dire quale scala sta usando.

# **Il gap di Bagheria è un'anomalia locale? Modello lineare di probabilità**
La domanda di policy del thread — gap locale o regionale — testata formalmente invece che
a occhio. GLM binomiale con link identità sui conteggi aggregati: è il modello lineare di
probabilità, quindi i coefficienti si leggono direttamente in punti percentuali.
L'interazione genere × territorio stima la differenza fra il gap di Bagheria e quello di
ciascun benchmark. Confronti pre-specificati, per non pescare a strascico: anno di
riferimento 2024, e pooled 2022-2024 come robustezza.

Caveat sul pooled: tre annualità contano più volte le stesse persone, quindi i suoi CI
sono un po' ottimisti. Il pooled serve a stabilizzare il punto, non a moltiplicare i dati.

In [7]:
import warnings

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import DomainWarning, PerfectSeparationWarning

# Il modello è saturo (una media per cella genere × territorio): il link identità fuori
# dal dominio [0,1] e la "separazione perfetta" sono attesi qui, non problemi di stima.
warnings.filterwarnings("ignore", category=DomainWarning)
warnings.filterwarnings("ignore", category=PerfectSeparationWarning)
warnings.filterwarnings("ignore", message="divide by zero", category=RuntimeWarning)

BENCHMARK = ["Palermo", "Sicilia", "Italia"]


def eccesso_di_gap(anni: list, periodo: str) -> pd.DataFrame:
    """LPM sui conteggi aggregati: gap di Bagheria meno gap di ciascun benchmark."""
    d = (giovani[giovani["genere"].isin(["M", "F"]) & giovani["anno"].isin(anni)]
         .groupby(["nome_territorio", "genere"], as_index=False)[["occupati", "popolazione"]].sum())
    d["p"] = d["occupati"] / d["popolazione"]
    modello = smf.glm("p ~ C(nome_territorio, Treatment('Bagheria')) * C(genere, Treatment('F'))",
                      data=d, family=sm.families.Binomial(link=sm.families.links.Identity()),
                      var_weights=d["popolazione"]).fit()
    righe = []
    for nome in BENCHMARK:
        chiave = (f"C(nome_territorio, Treatment('Bagheria'))[T.{nome}]"
                  ":C(genere, Treatment('F'))[T.M]")
        lo, hi = modello.conf_int().loc[chiave]
        # L'interazione stima gap_benchmark - gap_Bagheria: segno invertito per leggere
        # l'eccesso locale (positivo = il gap di Bagheria è più largo).
        righe.append({"periodo": periodo, "confronto": f"vs {nome}",
                      "eccesso gap Bagheria (pp)": -100 * modello.params[chiave],
                      "CI 95% basso": -100 * hi, "CI 95% alto": -100 * lo,
                      "p": modello.pvalues[chiave]})
    return pd.DataFrame(righe)


eccessi = pd.concat([eccesso_di_gap([2024], "2024"),
                     eccesso_di_gap([2022, 2023, 2024], "pooled 2022-2024")], ignore_index=True)
eccessi.round({"eccesso gap Bagheria (pp)": 1, "CI 95% basso": 1, "CI 95% alto": 1, "p": 3})

,periodo,confronto,eccesso gap Bagheria (pp),CI 95% basso,CI 95% alto,p
0,2024,vs Palermo,1.3,-0.4,3.1,0.130
1,2024,vs Sicilia,-1.6,-3.2,0.1,0.066
2,2024,vs Italia,-1.4,-3.0,0.3,0.107
3,pooled 2022-2024,vs Palermo,1.1,0.1,2.1,0.031
4,pooled 2022-2024,vs Sicilia,-1.8,-2.7,-0.8,0.000
5,pooled 2022-2024,vs Italia,-1.9,-2.9,-1.0,0.000


**📌 Risultato chiave** — Pooled 2022-2024, il gap di Bagheria è **+1.1 pp più largo di Palermo** (p=0.03) e **1.8-1.9 pp più stretto di Sicilia e Italia** (p<0.001); sul solo 2024 nessuna differenza è significativa. In punti percentuali il gap di Bagheria **non è un'anomalia locale**. Ma il "vantaggio" sulla Sicilia nasce dal tasso maschile più basso (16.5 contro 20.3), non da un tasso femminile migliore.

> Sul solo 2024 nessuna differenza fra gap è significativa al 5%: un anno singolo di un
> comune non ha la precisione per distinguerli. Sul pooled 2022-2024 il quadro si separa:
>
> - il gap di Bagheria è **più largo di quello di Palermo** (+1.1 punti, p=0.03);
> - ed è **più stretto di quello di Sicilia e Italia** (-1.8 e -1.9 punti, p<0.001).
>
> Quindi, in punti percentuali, il gap di genere di Bagheria non è un'anomalia locale: sta
> dentro il paesaggio regionale. Ma il "vantaggio" verso la Sicilia non è una buona
> notizia: nasce dal tasso maschile più basso (16.5 contro 20.3), non da un tasso femminile
> migliore. Messo insieme alla sezione sulle scale, il tratto distintivo di Bagheria non è
> l'ampiezza del divario: è il **livello** — le ragazze hanno il tasso di occupazione più
> basso del panel (8.2%) e lo svantaggio relativo più alto (M/F = 2.0). È questo il
> bersaglio per la proposal, non la chiusura di un gap "anomalo" che i dati non mostrano.

# **Il gap si sta allargando? Trend 2018-2024**
OLS sul gap in punti, anno centrato sul 2021, interazione col territorio per confrontare
le pendenze. Sei punti temporali per territorio e ogni punto è a sua volta una stima:
il modello dà direzione e ordine di grandezza, non inferenza fine.

In [8]:
serie_gap = ci_gap[["nome_territorio", "anno", "gap"]].assign(anno_c=lambda d: d["anno"] - 2021)
trend = smf.ols("gap ~ anno_c * C(nome_territorio, Treatment('Bagheria'))", data=serie_gap).fit()

pendenze = (pd.DataFrame({"pendenza (pp/anno)": trend.params, "p": trend.pvalues})
            .join(trend.conf_int().set_axis(["CI 95% basso", "CI 95% alto"], axis=1)))
pendenze = pendenze[pendenze.index.str.startswith("anno_c")]
pendenze.index = (pendenze.index
    .str.replace("anno_c:C(nome_territorio, Treatment('Bagheria'))[T.",
                 "differenza vs Bagheria: ", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("anno_c", "Bagheria", regex=False))
pendenze.round(3)[["pendenza (pp/anno)", "CI 95% basso", "CI 95% alto", "p"]]

,pendenza (pp/anno),CI 95% basso,CI 95% alto,p
Bagheria,0.205,0.060,0.350,0.008
differenza vs Bagheria: Italia,0.058,-0.147,0.262,0.558
differenza vs Bagheria: Palermo,0.030,-0.174,0.235,0.757
differenza vs Bagheria: Sicilia,0.252,0.047,0.456,0.019


**📌 Risultato chiave** — Il gap in punti di Bagheria cresce di **+0.21 pp/anno** (CI 0.06-0.35, p=0.008): l'allargamento è reale, non rumore. Il passo è indistinguibile da Palermo e Italia; solo la Sicilia allarga più in fretta (+0.25 pp/anno in più). Nello stesso periodo il rapporto M/F **scende** (2.47 → 2.01): le due scale divergono perché entrambi i tassi salgono.

> Il gap in punti di Bagheria cresce di ~0.2 punti l'anno (CI 0.06-0.35, p=0.008):
> l'allargamento è reale, non rumore. Il passo è indistinguibile da Palermo e Italia;
> solo la Sicilia allarga significativamente più in fretta (+0.25 punti/anno in più).
> Coerente con la sezione sulle scale: il divario in punti si allarga mentre quello
> relativo si riduce, perché entrambi i tassi stanno salendo e quello maschile sale di
> più in valore assoluto.

# **Dentro il "fuori da lavoro e studio": la popolazione per stato**
Il tasso di disoccupazione ha per denominatore le sole forze di lavoro, che per le
ragazze di Bagheria sono qualche centinaio di persone: è la misura più fragile del
thread e da sola non regge conclusioni. Al suo posto: la popolazione 15-24 ripartita
in quattro stati esaustivi — occupati, in cerca, studenti, altri inattivi — tutti su
denominatore-popolazione. La partizione è verificata contro i totali prima dell'uso;
per costruzione "in cerca + altri inattivi" coincide con la misura condivisa
`fuori_da_lavoro_e_studio`.

In [9]:
lavoro_15_24 = istr_lav[
    istr_lav["tavola"].eq("lavoro")
    & istr_lav["territorio"].isin(CONFRONTO)
    & istr_lav["eta"].eq("Y15-24")
    & istr_lav["cittadinanza"].eq("TOTAL")
    & istr_lav["titolo_studio"].eq("ALL")
    & istr_lav["genere"].isin(["M", "F", "T"])]
largo = lavoro_15_24.pivot_table(index=["territorio", "anno", "genere"],
                                 columns="condizione", values="valore", aggfunc="sum")

# Verifica della partizione. Tolleranza e non uguaglianza: i conteggi comunali non sono
# interi, il censimento permanente è una stima registro + campione.
DETTAGLIO = ["1", "12", "5", "4", "24", "7"]  # occupato, in cerca, studente, casalinga/o, pensione, altro
assert (largo[DETTAGLIO].sum(axis=1) - largo["99"]).abs().max() < 0.001, "la partizione non ricostruisce il totale"
assert (largo[["1", "12"]].sum(axis=1) - largo["22"]).abs().max() < 0.001, "occupati + in cerca != forze di lavoro"
per_genere = largo["99"].unstack("genere")
assert (per_genere["M"] + per_genere["F"] - per_genere["T"]).abs().max() < 0.001, "M + F != T"

stati = pd.DataFrame({
    "occupati": largo["1"], "in cerca": largo["12"], "studenti": largo["5"],
    "altri inattivi": largo[["4", "24", "7"]].sum(axis=1),
})
quote = (100 * stati.div(largo["99"], axis=0)).reset_index()
quote["nome_territorio"] = quote["territorio"].map(NOMI)

lungo = (quote.melt(id_vars=["territorio", "nome_territorio", "anno", "genere"],
                    var_name="stato", value_name="quota")
         .assign(quota=lambda d: d["quota"].round(1)))
lungo.to_csv(PROCESSED / "genere_composizione_stato.csv", index=False)

forze_f = largo.loc[(BAGHERIA, slice(None), "F"), "22"].droplevel([0, 2]).round(0).astype(int)
print("Forze di lavoro femminili 15-24 a Bagheria (denominatore del tasso di disoccupazione):")
print(forze_f.to_dict())
print("\nComposizione 2024 (% della popolazione 15-24):")
(lungo[lungo["anno"].eq(2024) & lungo["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns=["stato", "genere"], values="quota")
    .reindex(ORDINE)[["occupati", "in cerca", "studenti", "altri inattivi"]])

Forze di lavoro femminili 15-24 a Bagheria (denominatore del tasso di disoccupazione):
{2018: 676, 2019: 663, 2021: 432, 2022: 479, 2023: 528, 2024: 434}

Composizione 2024 (% della popolazione 15-24):


stato           occupati       in cerca      studenti       altri inattivi      
genere                 F     M        F    M        F     M              F     M
nome_territorio                                                                 
Bagheria             8.2  16.5      6.9  8.9     65.1  56.4           19.9  18.2
Palermo              9.6  16.5      7.2  9.3     66.8  59.8           16.4  14.3
Sicilia             10.4  20.3      6.6  8.3     67.8  57.0           15.2  14.4
Italia              17.3  26.9      5.9  6.4     67.8  57.4            9.0   9.3

**📌 Risultato chiave** — Il minor tasso di occupazione femminile a 15-24 è assorbito dallo **studio**, non dall'inattività: studenti 65.1% F contro 56.4% M, in cerca 6.9% contro 8.9%, altri inattivi 19.9% contro 18.2%. E sotto il gap di genere ce n'è uno territoriale che colpisce **entrambi** i generi: "altri inattivi" al 14-20% a Bagheria, Palermo e Sicilia contro il ~9% nazionale.

> Tre letture:
>
> 1. Le ragazze **non** sono più spesso "fuori da tutto" dei ragazzi: altri inattivi
>    19.9% contro 18.2%, in cerca 6.9% contro 8.9%. Il tasso di occupazione più basso è
>    assorbito quasi per intero da più studio (65.1% contro 56.4%). A 15-24 il gap
>    occupazionale fotografa soprattutto ragazze ancora nel sistema formativo, non
>    inattività femminile.
> 2. Ciò che separa Bagheria, Palermo e la Sicilia dall'Italia è la quota di "altri
>    inattivi" per **entrambi** i generi: ~14-20% contro il ~9% nazionale.
> 3. Il denominatore del tasso di disoccupazione femminile di Bagheria è di 430-680
>    persone: i suoi sbalzi annuali sono in gran parte rumore. Le conclusioni del thread
>    usano occupazione e composizione, non quel tasso.
>
> Il punto 1 sposta la domanda di policy in avanti: se a 15-24 le ragazze studiano di più
> e il vantaggio non si converte poi in occupazione, il nodo sta all'uscita dal percorso
> formativo — che i dati comunali non permettono di osservare oltre i 24 anni (la classe
> successiva, 25-49, sfora il target giovani e non è scomponibile).

# **Dentro gli "altri inattivi": casalinghe a 15-24 anni**
La decomposizione sopra mostra "altri inattivi" quasi uguali fra i generi (19.9% F contro
18.2% M nel 2024). È un'uguaglianza apparente: i codici che la compongono — casalinga/o,
percettore/rice di pensione, altra condizione — hanno distribuzioni di genere opposte,
e tenerli aggregati nasconde il meccanismo.

In [10]:
# Scissione degli "altri inattivi" nei tre codici che li compongono.
inattivi_dettaglio = pd.DataFrame({
    "casalinghe_o_i": largo["4"],
    "percettori_pensione": largo["24"],
    "altra_condizione": largo["7"],
})
quote_inattivi = (100 * inattivi_dettaglio.div(largo["99"], axis=0)).round(1)
casalinghe = pd.concat([inattivi_dettaglio["casalinghe_o_i"].rename("conteggio"),
                        quote_inattivi.add_suffix("_%")], axis=1).reset_index()
casalinghe["nome_territorio"] = casalinghe["territorio"].map(NOMI)
casalinghe.to_csv(PROCESSED / "genere_casalinghe.csv", index=False)

# Versione a sei stati della composizione, per la figura: qui "altri inattivi" è scisso,
# mentre genere_composizione_stato.csv resta a quattro stati per gli altri thread.
sei_stati = pd.DataFrame({
    "occupati": largo["1"], "in cerca": largo["12"], "studenti": largo["5"],
    "casalinghe/i": largo["4"], "altra condizione": largo["7"], "pensione": largo["24"],
})
dettaglio = (100 * sei_stati.div(largo["99"], axis=0)).round(1).reset_index()
dettaglio["nome_territorio"] = dettaglio["territorio"].map(NOMI)
(dettaglio.melt(id_vars=["territorio", "nome_territorio", "anno", "genere"],
                var_name="stato", value_name="quota")
    .to_csv(PROCESSED / "genere_composizione_stato_dettaglio.csv", index=False))

serie_bag = (casalinghe[casalinghe["territorio"].eq(BAGHERIA) & casalinghe["genere"].eq("F")]
             .set_index("anno")[["conteggio", "casalinghe_o_i_%"]]
             .assign(conteggio=lambda d: d["conteggio"].round().astype(int)))
print("Bagheria — ragazze 15-24 che si dichiarano casalinghe:")
print(serie_bag.to_string())
print("\n2024, quote sulla popolazione 15-24 (%):")
(casalinghe[casalinghe["anno"].eq(2024) & casalinghe["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere",
                 values=["casalinghe_o_i_%", "altra_condizione_%"])
    .reindex(ORDINE)[["casalinghe_o_i_%", "altra_condizione_%"]])

Bagheria — ragazze 15-24 che si dichiarano casalinghe:
      conteggio  casalinghe_o_i_%
anno                             
2018        376              12.4
2019        324              10.9
2021        421              14.8
2022        364              12.8
2023        421              14.6
2024        387              13.4

2024, quote sulla popolazione 15-24 (%):


casalinghe_o_i_%      altra_condizione_%      
genere                         F    M                  F     M
nome_territorio                                               
Bagheria                    13.4  1.7                6.4  16.1
Palermo                     11.3  1.4                5.0  12.5
Sicilia                     10.1  1.2                5.1  12.7
Italia                       4.6  0.6                4.4   8.5

**📌 Risultato chiave** — Il **13.4% delle ragazze 15-24 di Bagheria si dichiara casalinga (387 persone)** contro l'1.7% dei ragazzi e il 4.6% delle coetanee italiane: quasi il triplo dell'incidenza nazionale. Gradiente territoriale netto (Italia 4.6 < Sicilia 10.1 < Palermo 11.3 < Bagheria 13.4) e serie 2018-2024 sempre fra 320 e 420 ragazze: è **strutturale**. Per i maschi gli "altri inattivi" sono invece quasi tutti "in altra condizione" (16.1%).

> L'uguaglianza era apparente: **il 13.4% delle ragazze 15-24 di Bagheria si dichiara
> casalinga (387 persone) contro l'1.7% dei ragazzi**; per i maschi gli "altri inattivi"
> sono quasi tutti "in altra condizione". E il fenomeno ha un gradiente territoriale
> netto — Italia 4.6%, Sicilia 10.1%, Palermo 11.3%, Bagheria 13.4%: il triplo
> dell'incidenza nazionale. La serie 2018-2024 (sempre fra 320 e 420 ragazze) dice che
> è strutturale, non episodico.
>
> È il carico di cura reso visibile nei dati, ed è un'evidenza da proposal: un target
> definito (le ~390 ragazze casalinghe), un meccanismo nominabile e un KPI naturale
> (la quota casalinghe 15-24, da portare prima al livello di Palermo, poi verso quello
> nazionale). Cautela: è la condizione autodichiarata al censimento, non una misura del
> lavoro di cura effettivo — va usata come marcatore del fenomeno, non come sua stima.

# **Gap di genere sull'istruzione, 9-24**
Composizione per titolo di studio e quota con almeno il diploma.

In [11]:
# Categorie mutuamente esclusive: la loro somma torna esattamente al totale ALL (verificato sotto).
ALMENO_DIPLOMA = ["USE_IF", "BL", "ML_RDD"]   # diploma, terziario 1° livello, terziario 2° e dottorato
TITOLI = ["NED", "PSE", "LSE"] + ALMENO_DIPLOMA

scuola = istruzione[
    istruzione["territorio"].isin(CONFRONTO)
    & istruzione["eta"].eq("Y9-24")
    & istruzione["cittadinanza"].eq("TOTAL")
    & istruzione["genere"].isin(["M", "F", "T"])]

conteggi = scuola.pivot_table(index=["territorio", "anno", "genere"],
                              columns="titolo_studio", values="valore", aggfunc="sum")

# Controllo di coerenza: le categorie di dettaglio devono ricostruire il totale.
scarto = (conteggi[TITOLI].sum(axis=1) - conteggi["ALL"]).abs().max()
assert scarto == 0, f"le categorie non ricostruiscono il totale, scarto massimo {scarto}"

titoli = pd.DataFrame({
    "popolazione_9_24": conteggi["ALL"],
    "almeno_diploma_%": 100 * conteggi[ALMENO_DIPLOMA].sum(axis=1) / conteggi["ALL"],
    "licenza_media_%": 100 * conteggi["LSE"] / conteggi["ALL"],
    "nessun_titolo_o_elementare_%": 100 * conteggi[["NED", "PSE"]].sum(axis=1) / conteggi["ALL"],
}).round(1).reset_index()
titoli["nome_territorio"] = titoli["territorio"].map(NOMI)
titoli.to_csv(PROCESSED / "genere_istruzione.csv", index=False)

ultimo_anno = titoli["anno"].max()
(titoli[titoli["anno"].eq(ultimo_anno) & titoli["genere"].isin(["M", "F"])]
    .pivot(index="nome_territorio", columns="genere", values="almeno_diploma_%")
    .reindex(ORDINE)
    .assign(**{"gap M-F (punti)": lambda d: (d["M"] - d["F"]).round(1)}))

genere,F,M,gap M-F (punti)
nome_territorio,,,
Bagheria,33.4,29.2,-4.2
Palermo,30.5,28.7,-1.8
Sicilia,33.1,30.4,-2.7
Italia,34.5,32.3,-2.2


**📌 Risultato chiave** — Gap istruzione 9-24 = **-4.2 pp**: a Bagheria le ragazze arrivano almeno al diploma più dei coetanei (33.4% contro 29.2%), ed è il vantaggio femminile più ampio del panel (Palermo -1.8, Sicilia -2.7, Italia -2.2).

> Il gap sull'istruzione è **negativo**: le ragazze arrivano al diploma più spesso dei coetanei,
> a Bagheria come altrove.
>
> Il livello assoluto va letto con cautela — la fascia parte da 9 anni e include ragazzi che il
> diploma non possono ancora averlo, quindi la percentuale è strutturalmente bassa. Il confronto
> fra generi resta valido, perché il denominatore è distorto allo stesso modo per entrambi.

# **Verifica di composizione per età**
I tassi delle fasce 15-24 e 9-24 sono aggregati: se la struttura per età dentro la fascia
differisse fra generi o fra territori, parte dei gap sarebbe un artefatto di composizione
(chi ha 15 anni non lavora quasi mai, chi ne ha 12 non può avere un diploma). Le età
singole della demografia, disponibili dal 2021, permettono il controllo sul 2024.

In [12]:
singole_2024 = popolazione[
    popolazione["territorio"].isin(CONFRONTO)
    & popolazione["cittadinanza"].eq("TOTAL")
    & popolazione["genere"].isin(["M", "F"])
    & popolazione["eta_anni"].notna()
    & popolazione["anno"].eq(2024)]


def quota_fascia_alta(fascia, alta):
    """% della sottofascia più vecchia dentro la fascia, per territorio e genere (2024)."""
    dentro = singole_2024[singole_2024["eta_anni"].between(*fascia)]
    totale = dentro.groupby(["territorio", "genere"])["valore"].sum()
    parte = dentro[dentro["eta_anni"].between(*alta)].groupby(["territorio", "genere"])["valore"].sum()
    return (100 * parte / totale).unstack().rename(index=NOMI).reindex(ORDINE).round(1)


print("Fascia lavoro — quota dei 20-24 dentro i 15-24:")
print(quota_fascia_alta((15, 24), (20, 24)).to_string())
print("\nFascia istruzione — quota dei 19-24 dentro i 9-24:")
quota_fascia_alta((9, 24), (19, 24))

Fascia lavoro — quota dei 20-24 dentro i 15-24:
genere         F     M
territorio            
Bagheria    52.0  50.7
Palermo     49.2  49.6
Sicilia     50.8  50.6
Italia      50.2  50.7

Fascia istruzione — quota dei 19-24 dentro i 9-24:


genere,F,M
territorio,,
Bagheria,40.8,38.3
Palermo,38.5,38.7
Sicilia,39.5,40.1
Italia,38.8,39.7


**📌 Risultato chiave** — Il gap occupazionale **non** è un artefatto di composizione per età: le ragazze 15-24 di Bagheria sono semmai il gruppo più "vecchio" del panel (52.0% ha 20-24 anni), il che dovrebbe alzarne il tasso — il controllo è conservativo rispetto alla conclusione. Sul gap istruzione invece **~1.5 dei 4.2 punti sono mix per età**: il segno regge, la magnitudine di Bagheria va citata con questa cautela.

> - **Fascia lavoro 15-24**: la quota di 20-24enni sta fra il 49% e il 52% ovunque, per
>   entrambi i generi. Semmai le ragazze di Bagheria sono il gruppo leggermente più
>   "vecchio" (52.0%), il che dovrebbe *alzarne* il tasso di occupazione: il livello
>   femminile più basso del panel non è un artefatto di composizione, e il controllo è
>   conservativo rispetto alla conclusione.
> - **Fascia istruzione 9-24**: a Bagheria le femmine sono più concentrate nei 19-24 dei
>   maschi (40.8% contro 38.3%), mentre negli altri territori la differenza è sotto il
>   punto e spesso di segno opposto. Parte del gap di istruzione di Bagheria (-4.2) è
>   quindi composizione: assumendo che il diploma stia quasi tutto nei 19-24, l'ordine di
>   grandezza è 2.5 punti di mix × ~60 punti di differenza fra le sottofasce ≈ **1.5 punti
>   dei 4.2**. Il segno del vantaggio femminile regge (c'è anche dove il mix è pari), la
>   magnitudine di Bagheria va citata con questa cautela. La standardizzazione esatta non
>   è possibile: il titolo di studio per età singola non esiste a livello comunale.

# **I due gap a confronto**
La domanda del thread, nella forma che i dati permettono: le ragazze studiano di più e lavorano
di meno, e questo distingue Bagheria dai territori di riferimento?

In [13]:
anno_comune = min(giovani["anno"].max(), titoli["anno"].max())

occupazione_gap = (giovani[giovani["anno"].eq(anno_comune) & giovani["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere", values="tasso_occupazione"))
istruzione_gap = (titoli[titoli["anno"].eq(anno_comune) & titoli["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere", values="almeno_diploma_%"))

quadro = pd.DataFrame({
    "occupazione M": occupazione_gap["M"],
    "occupazione F": occupazione_gap["F"],
    "gap occupazione (M-F)": occupazione_gap["M"] - occupazione_gap["F"],
    "almeno diploma M": istruzione_gap["M"],
    "almeno diploma F": istruzione_gap["F"],
    "gap istruzione (M-F)": istruzione_gap["M"] - istruzione_gap["F"],
}).reindex(ORDINE).round(1)
quadro.to_csv(PROCESSED / "genere_quadro_sintesi.csv")

print(f"Anno: {anno_comune}   —   occupazione 15-24, istruzione 9-24")
quadro

Anno: 2024   —   occupazione 15-24, istruzione 9-24


,occupazione M,occupazione F,gap occupazione (M-F),almeno diploma M,almeno diploma F,gap istruzione (M-F)
nome_territorio,,,,,,
Bagheria,16.5,8.2,8.3,29.2,33.4,-4.2
Palermo,16.5,9.6,6.9,28.7,30.5,-1.8
Sicilia,20.3,10.4,9.9,30.4,33.1,-2.7
Italia,26.9,17.3,9.6,32.3,34.5,-2.2


**📌 Risultato chiave** — Il paradosso, nella sua forma più compatta: **istruzione -4.2 pp (a favore delle ragazze), occupazione +8.3 pp (a loro sfavore)**, e i due segni sono opposti in tutti e quattro i territori. Le ragazze di Bagheria studiano più dei coetanei e lavorano la metà.

> È il risultato centrale del thread: **il gap di istruzione favorisce le ragazze, quello
> sull'occupazione le penalizza**, e i due segni sono opposti in tutti e quattro i territori.
>
> Per la proposal conta il confronto, non il livello: se il gap occupazionale di Bagheria è vicino
> a quello siciliano, il problema è regionale e un intervento comunale può poco. Se è più largo,
> c'è qualcosa di locale su cui agire.
>
> La risposta formale sta nella sezione LPM sopra: in punti percentuali il gap di Bagheria non è
> un'anomalia (sopra Palermo, sotto Sicilia e Italia). Lo specifico locale è il **livello**
> dell'occupazione femminile, il più basso del panel — ed è quello, insieme al conteggio della
> sezione "Il gap in persone" qui sotto, che la proposal deve bersagliare.

# **Il gap in persone**
Per il template della proposal (target e KPI) i tassi vanno tradotti in teste: quante
ragazze 15-24 occupate in più servirebbero a Bagheria sotto tre tassi-obiettivo. Calcolo
sui conteggi non arrotondati; la colonna "media 2022-2024" è la versione robusta alla
scelta dell'anno. Sono stock annui, non cumulati: "+40" significa 40 occupate in più
nell'anno di riferimento, a parità di popolazione.

In [14]:
def occupate_in_piu(anni: list) -> pd.Series:
    """Quante ragazze 15-24 occupate in più a Bagheria sotto ciascun tasso-obiettivo."""
    d = (giovani[giovani["genere"].isin(["M", "F"]) & giovani["anno"].isin(anni)]
         .groupby(["nome_territorio", "genere"])[["occupati", "popolazione"]].sum())
    p = d["occupati"] / d["popolazione"]
    pop_f = d.loc[("Bagheria", "F"), "popolazione"] / len(anni)  # popolazione media annua
    base = p[("Bagheria", "F")]
    return pd.Series({
        "parità con i coetanei maschi di Bagheria": int(round(pop_f * (p[("Bagheria", "M")] - base))),
        "tasso femminile di Palermo": int(round(pop_f * (p[("Palermo", "F")] - base))),
        "tasso femminile dell'Italia": int(round(pop_f * (p[("Italia", "F")] - base))),
    })


persone = pd.DataFrame({"occupate in più (2024)": occupate_in_piu([2024]),
                        "occupate in più (media 2022-2024)": occupate_in_piu([2022, 2023, 2024])})
persone.index.name = "scenario"
persone.to_csv(PROCESSED / "genere_gap_persone.csv")

base_2024 = giovani.query("nome_territorio == 'Bagheria' and genere == 'F' and anno == 2024")
print(f"Base 2024: {base_2024['popolazione'].item():.0f} ragazze 15-24, "
      f"{base_2024['occupati'].item():.0f} occupate "
      f"({base_2024['occupati'].item() / base_2024['popolazione'].item():.1%})")
persone

Base 2024: 2882 ragazze 15-24, 236 occupate (8.2%)


,occupate in più (2024),occupate in più (media 2022-2024)
scenario,,
parità con i coetanei maschi di Bagheria,239,221
tasso femminile di Palermo,40,34
tasso femminile dell'Italia,262,255


**📌 Risultato chiave** — Base 2024: **2.882 ragazze 15-24 a Bagheria, 236 occupate (8.2%)**. Allinearsi al tasso femminile di Palermo vale **+40 occupate** (KPI realistico a 2-3 anni); la parità con i coetanei o il tasso nazionale valgono **+239 / +262**, cioè più che raddoppiare le occupate attuali — misura del problema, non obiettivo.

> La scala dell'intervento, in persone:
>
> - allinearsi al tasso femminile di **Palermo** vale **+40 occupate**: è l'obiettivo che
>   un intervento comunale può realisticamente rivendicare come KPI a 2-3 anni;
> - la **parità con i coetanei** o il **tasso femminile nazionale** valgono +239 e +262:
>   più che raddoppiare le 236 occupate attuali. È la misura del problema, non un KPI.
>
> Per il template della proposal: l'evidenza è questa tabella (più i CI della sezione
> sugli intervalli), il target sono le ~2.900 ragazze 15-24 di Bagheria, il KPI sono i
> punti di tasso chiusi verso il benchmark scelto — e ogni cifra si rigenera da questa
> cella a ogni aggiornamento dei dati.

# **Chi se ne va? Ritenzione di coorte per genere, 2021-2024**
La fuga di talenti del bando, guardata per genere. Dalle età singole si segue ogni
coorte: chi aveva `a` anni nel 2021 ne ha `a+3` nel 2024, e il rapporto fra i due stock
è la ritenzione netta. A queste età la mortalità è trascurabile, quindi lo scarto da 100
è migrazione netta — impastata però con l'aggiustamento post-censuario delle stime, che
non è separabile: si leggono i pattern contro il benchmark nazionale, non i decimali.
"Netta" significa che conta anche chi arriva, non solo chi parte.

In [15]:
singole = (popolazione[
        popolazione["territorio"].isin(CONFRONTO)
        & popolazione["cittadinanza"].eq("TOTAL")
        & popolazione["genere"].isin(["M", "F"])
        & popolazione["eta_anni"].notna()]
    .assign(eta=lambda d: d["eta_anni"].astype(int)))

p21 = singole[singole["anno"].eq(2021)]
p24 = singole[singole["anno"].eq(2024)]

COORTI = {"15-19 nel 2021": (15, 19), "20-24 nel 2021": (20, 24), "25-29 nel 2021": (25, 29)}
righe = []
for etichetta, (a0, a1) in COORTI.items():
    base = p21[p21["eta"].between(a0, a1)].groupby(["territorio", "genere"])["valore"].sum()
    dopo = p24[p24["eta"].between(a0 + 3, a1 + 3)].groupby(["territorio", "genere"])["valore"].sum()
    parziale = (100 * dopo / base).rename("ritenzione_%").reset_index()
    parziale["coorte"] = etichetta
    righe.append(parziale)
coorti = pd.concat(righe, ignore_index=True)
coorti["nome_territorio"] = coorti["territorio"].map(NOMI)
coorti["ritenzione_%"] = coorti["ritenzione_%"].round(1)
coorti.to_csv(PROCESSED / "genere_coorti.csv", index=False)

(coorti.pivot_table(index="coorte", columns=["nome_territorio", "genere"], values="ritenzione_%")
    .reindex(columns=pd.MultiIndex.from_product([ORDINE, ["F", "M"]])))

Bagheria        Palermo        Sicilia        Italia       
                      F      M       F      M       F      M      F      M
coorte                                                                    
15-19 nel 2021    100.4   98.5   100.8  101.1   100.7  103.0  101.8  104.8
20-24 nel 2021    102.0   99.7    99.8   97.7    99.2   98.3  102.7  104.1
25-29 nel 2021     96.3  101.2    98.8   96.9    97.6   97.6  103.0  103.8

**📌 Risultato chiave** — Ritenzione di coorte 2021-2024: Bagheria sta **sotto l'Italia in ogni coorte e per entrambi i generi**. La firma di genere sta nel *quando*: i ragazzi si perdono presto (15-19: 98.5 contro 104.8), le ragazze **dopo i 25** (25-29: 96.3 contro 103.0, la cella peggiore del comune) — proprio quando il vantaggio formativo dovrebbe convertirsi in lavoro.

> Due pattern, letti contro il benchmark nazionale (che sta sopra 100 ovunque grazie
> all'immigrazione):
>
> 1. **Bagheria non raggiunge il livello nazionale in nessuna cella**: ogni coorte, di
>    entrambi i generi, trattiene meno giovani di quanto faccia l'Italia. Il drenaggio
>    riguarda tutti.
> 2. La firma di genere sta nel **quando**: i ragazzi si perdono presto (coorte 15-19
>    nel 2021: 98.5 contro il 104.8 nazionale), le ragazze **dopo i 25** — la coorte
>    25-29 femminile è la peggiore di Bagheria (96.3 contro 103.0, quasi 7 punti sotto).
>    È l'età in cui il percorso formativo finisce: coerente con la decomposizione per
>    stato, le ragazze restano finché studiano e il territorio ne perde una quota
>    proprio quando il vantaggio educativo dovrebbe convertirsi in lavoro.
>
> La misura è netta e su una finestra di tre anni: non distingue chi parte da chi
> arriva, né dice dove vanno. Per i flussi origine-destinazione servono altre fonti —
> il thread mobilità è il posto naturale dove cercarle.

# **Contesto storico 2011**
Gli indicatori di genere di 8milaCensus. Sono calcolati sui **15 anni e più**, non sui giovani:
servono come sfondo, non come termine di paragone con le serie qui sopra.

In [16]:
GENERE_2011 = ["L1", "L2", "L6", "L7", "L10", "L11", "I1"]

storico = (ottomila[
        ottomila["territorio"].isin(CONFRONTO)
        & ottomila["anno"].eq(2011)
        & ottomila["indicatore"].isin(GENERE_2011)]
    .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore")
    .pivot(index=["indicatore", "nome_indicatore"], columns="territorio", values="valore"))
storico = storico[CONFRONTO].rename(columns={c: NOMI[c] for c in CONFRONTO}).round(1)
storico

,territorio,Bagheria,Palermo,Sicilia,Italia
indicatore,nome_indicatore,,,,
I1,Differenziali di genere per l'istruzione superiore,98.9,102.5,100.7,101.5
L1,Partecipazione al mercato del lavoro maschile,56.7,57.8,57.5,60.7
L10,Tasso di occupazione maschile,43.1,45.0,46.9,54.8
L11,Tasso di occupazione femminile,18.1,25.5,24.0,36.1
L2,Partecipazione al mercato del lavoro femminile,28.7,35.9,33.0,41.8
L6,Tasso di disoccupazione maschile,24.1,22.1,18.5,9.8
L7,Tasso di disoccupazione femminile,36.9,29.1,27.1,13.6


**📌 Risultato chiave** — Sfondo di lungo periodo (2011, 15+, fonte diversa): tasso di occupazione femminile a Bagheria **18.1% contro 36.1% nazionale**, la metà, e disoccupazione femminile 36.9% contro 13.6%. Lo svantaggio femminile locale non è un fatto recente. Non confrontabile con le serie 2018-2024: fascia e fonte diverse, citare sempre con entrambe.

> `L10`/`L11` danno il gap occupazionale complessivo del 2011, `L1`/`L2` quello sulla
> partecipazione. Non sono confrontabili con le serie 2018-2024 di questo notebook: fascia
> diversa (15+ contro 15-24) e fonte diversa. Vanno citati con anno e fascia, sempre.

# **Bagheria nella distribuzione siciliana**
I quattro territori di confronto dicono se Bagheria è sopra o sotto la media, non dove si
colloca fra i comuni. L'unica fonte che copre **tutti i comuni siciliani** con un dato di
genere è 8milaCensus: indicatore `L11`, tasso di occupazione femminile, **2011, 15 anni e
più**. Fascia e anno diversi dalle serie di questo notebook — è un ritratto del contesto,
mai da accostare al 15-24 del censimento permanente.

Prepara la tabella per la mappa: valori uniti ai confini comunali (`pipeline/build.py`,
confini ISTAT 2026 generalizzati, EPSG:32633).

In [17]:
# I vertici arrivano già proiettati da pipeline/build.py: qui si uniscono ai valori e basta.
poligoni = pd.read_csv(PROCESSED / "comuni_sicilia_poligoni.csv",
                       dtype={"territorio": str, "nome_comune": str})
centroidi = pd.read_csv(PROCESSED / "comuni_sicilia_centroidi.csv",
                        dtype={"territorio": str, "nome_comune": str})

occ_femminile = (ottomila[
        ottomila["anno"].eq(2011)
        & ottomila["indicatore"].eq("L11")
        & ottomila["livello"].eq("1")][["territorio", "valore"]]
    .rename(columns={"valore": "occupazione_femminile_2011"}))

# Left join dai confini: un comune senza dato resta nella mappa come area vuota, non sparisce.
mappa = poligoni.merge(occ_femminile, on="territorio", how="left")
scoperti = sorted(set(poligoni["territorio"]) - set(occ_femminile["territorio"]))
mancanti = sorted(set(occ_femminile["territorio"]) - set(poligoni["territorio"]))
assert not mancanti, f"comuni con dato 2011 privi di confine: {mancanti}"
mappa.to_csv(PROCESSED / "genere_mappa_occupazione_femminile.csv", index=False)

# Posizione di Bagheria nella distribuzione dei 390 comuni.
valori = occ_femminile.set_index("territorio")["occupazione_femminile_2011"]
bagheria = valori[BAGHERIA]
percentile = 100 * (valori < bagheria).mean()
posizione = int((valori < bagheria).sum()) + 1

etichette = centroidi[centroidi["territorio"].isin([BAGHERIA, PALERMO])].copy()
etichette["valore"] = etichette["territorio"].map(valori)
etichette.to_csv(PROCESSED / "genere_mappa_etichette.csv", index=False)

print(f"comuni siciliani con dato 2011: {len(valori)}   confini disponibili: {poligoni['territorio'].nunique()}")
print("senza dato 2011:", [centroidi.set_index('territorio').loc[c, 'nome_comune'] for c in scoperti] or "nessuno")
print(f"\nBagheria: {bagheria:.1f}%  →  {posizione}° comune su {len(valori)} in ordine crescente "
      f"({percentile:.0f}° percentile)")
print(f"mediana siciliana {valori.median():.1f}%   min {valori.min():.1f}%   max {valori.max():.1f}%")

comuni siciliani con dato 2011: 390   confini disponibili: 391
senza dato 2011: ['Misiliscemi']

Bagheria: 18.1%  →  49° comune su 390 in ordine crescente (12° percentile)
mediana siciliana 23.6%   min 13.0%   max 39.4%


**📌 Risultato chiave** — Bagheria è nel **12° percentile siciliano** per occupazione femminile: solo 48 comuni su 390 stavano più in basso nel 2011 (49ª posizione in ordine crescente), contro una mediana regionale del 23.6%. Non è un comune medio della Sicilia, è nella coda bassa di una regione già ultima in Italia. Dato 2011, 15+, fonte 8milaCensus: contesto, non confrontabile con le serie 15-24 di questo notebook.

> Due avvertenze per l'uso in mappa. **Misiliscemi** (istituito nel 2021 staccandosi da
> Trapani) non ha un dato 2011 e resta in bianco: nel 2011 il suo territorio era dentro
> Trapani, e attribuirgli il valore trapanese sarebbe un'imputazione, non un dato.
> I **confini sono al 2026** mentre il dato è 2011: per tutti gli altri 390 comuni la
> corrispondenza è verificata dal join qui sopra (nessun comune con dato resta senza
> confine), e ISTAT non pubblica più le annate storiche su quello storage.

# **Sintesi finale**

I risultati delle sezioni precedenti, ricomposti nell'ordine in cui reggono la proposal.
Il riferimento fra virgolette è la sezione che produce la cifra: nulla qui è calcolato a
mano, tutto si rigenera rieseguendo il notebook.

---

### 1. Il paradosso istruzione / lavoro — il risultato centrale
* **Istruzione (9-24)**: 33.4% F contro 29.2% M, gap **-4.2 pp** — le ragazze studiano di più *(«Gap di genere sull'istruzione»)*.
* **Occupazione (15-24)**: 8.2% F contro 16.5% M, gap **+8.3 pp** — lavorano la metà *(«Gap di genere sull'occupazione»)*.
* **Meccanismo**: a 15-24 la differenza è assorbita dalla permanenza nello studio (65.1% F contro 56.4% M), non dall'inattività *(«Dentro il "fuori da lavoro e studio"»)*.
* I due segni sono **opposti in tutti e quattro i territori**: il problema non è che le ragazze si formino meno, è che il vantaggio formativo non si converte.

---

### 2. Il problema non è l'ampiezza del gap: è il livello
* **In punti** (8.3 pp) Bagheria sta sopra Palermo (6.9) ma sotto Sicilia (9.9) e Italia (9.7); il modello LPM lo conferma formalmente sul pooled 2022-2024 (+1.1 vs Palermo, p=0.03; -1.8/-1.9 vs Sicilia/Italia, p<0.001) *(«Il gap di Bagheria è un'anomalia locale?»)*.
* **In rapporto M/F** (2.01) Bagheria è invece la **peggiore del panel** (Palermo 1.72, Sicilia 1.95, Italia 1.56) *(«Punti percentuali o rapporto?»)*.
* Il bersaglio della proposal è quindi il **livello dell'occupazione femminile — 8.2%, il più basso dei quattro territori** — non la chiusura di un divario "anomalo" che i dati non mostrano.

---

### 3. Il carico di cura, reso visibile
* Il **13.4% delle ragazze 15-24 (387 persone)** si dichiara casalinga, contro l'1.7% dei coetanei e il 4.6% delle coetanee italiane *(«Dentro gli "altri inattivi"»)*.
* Gradiente territoriale netto (Italia 4.6 < Sicilia 10.1 < Palermo 11.3 < Bagheria 13.4) e serie 2018-2024 stabile fra 320 e 420 ragazze: **fenomeno strutturale, non episodico**.
* È l'unico meccanismo del thread con un target nominabile e un KPI naturale già pronto.

---

### 4. Il gap si allarga — lentamente, ma davvero
* +0.21 pp/anno a Bagheria (CI 0.06-0.35, p=0.008), passo indistinguibile da Palermo e Italia *(«Il gap si sta allargando?»)*.
* Nello stesso periodo il rapporto M/F **scende** (2.47 → 2.01): entrambi i tassi salgono, quello maschile di più in valore assoluto. Ogni claim deve dichiarare quale delle due scale sta usando.

---

### 5. La fuga di talenti ha un tempismo di genere
* Bagheria trattiene meno dell'Italia in **ogni** coorte e per entrambi i generi: il drenaggio riguarda tutti *(«Ritenzione di coorte per genere»)*.
* Ma i ragazzi si perdono a 15-19 (98.5 contro 104.8), le ragazze **dopo i 25** (96.3 contro 103.0). Le ragazze restano finché studiano; il territorio le perde all'uscita dal percorso formativo — esattamente il punto in cui il vantaggio educativo dovrebbe diventare lavoro.

---

### 6. Sotto il gap di genere ce n'è uno territoriale
* La quota di "altri inattivi" 15-24 è 14-20% a Bagheria, Palermo e Sicilia contro il ~9% nazionale, **per entrambi i generi** *(«Dentro il "fuori da lavoro e studio"»)*.
* E il 2011 mostra radici lunghe: occupazione femminile 15+ al 18.1% contro il 36.1% nazionale *(«Contesto storico 2011»)*.
* Un intervento di genere agisce quindi su uno svantaggio doppio: essere giovane a Bagheria, ed esservi giovane donna.

---

### 7. Traduzione in persone (per il template della proposal)
Base 2024: **2.882 ragazze 15-24, 236 occupate** *(«Il gap in persone»)*.

| Scenario | Occupate in più | Uso |
|---|---|---|
| Tasso femminile di Palermo | **+40** | KPI realistico a 2-3 anni |
| Parità con i coetanei maschi | +239 | misura del problema |
| Tasso femminile nazionale | +262 | misura del problema |

---

### Cosa entra nella proposal, e con quale KPI
L'intervento è la colonna che il team deve riempire: qui ci sono solo evidenza, target e
KPI, che sono fatti misurabili e si rigenerano da questo notebook.

| Evidenza (sezione) | Target | KPI misurabile |
|---|---|---|
| Occupazione femminile 15-24 all'8.2%, la più bassa del panel, con rapporto M/F 2.01 *(«Punti percentuali o rapporto?», «Il gap in persone»)* | le 2.882 ragazze 15-24 residenti | +40 occupate = tasso femminile allineato a Palermo (9.6%) entro 2-3 anni |
| 387 ragazze 15-24 casalinghe, 13.4% contro il 4.6% nazionale, stabile dal 2018 *(«Dentro gli "altri inattivi"»)* | le ~390 casalinghe 15-24 | quota casalinghe 15-24: prima al livello di Palermo (11.3%), poi verso quello nazionale |
| Ritenzione femminile 25-29 al 96.3 contro il 103.0 nazionale *(«Ritenzione di coorte»)* | le coorti femminili in uscita dal percorso formativo | ritenzione netta 25-29 F portata almeno al livello di Palermo (98.8) |

### Cosa resta nell'analisi e non entra nella proposal
* **Decomposizione del gap per titolo di studio**: non calcolabile a livello comunale *(«Verifica di fattibilità»)*.
* **Tasso di disoccupazione femminile**: denominatore di 430-680 persone, oscillazioni in larga parte rumore *(«Dentro il "fuori da lavoro e studio"»)*.
* **Magnitudine del gap di istruzione**: ~1.5 dei 4.2 punti sono composizione per età; il segno regge, il numero va citato con la cautela *(«Verifica di composizione per età»)*.
* **Indicatori 2011**: sfondo storico su 15+, mai in serie con il 2018-2024 *(«Contesto storico 2011»)*.


# **Tabelle esportate**
Le figure R di questo thread leggono da qui. Nessun ricalcolo in R.

In [18]:
pd.DataFrame(
    [(p.name, sum(1 for _ in p.open()) - 1) for p in sorted(PROCESSED.glob("genere_*.csv"))],
    columns=["file", "righe"])

,file,righe
0,genere_casalinghe.csv,72
1,genere_composizione_stato.csv,288
2,genere_composizione_stato_dettaglio.csv,432
3,genere_coorti.csv,24
4,genere_gap_occupazione.csv,24
5,genere_gap_occupazione_ci.csv,24
6,genere_gap_persone.csv,3
7,genere_istruzione.csv,84
8,genere_mappa_etichette.csv,2
9,genere_mappa_occupazione_femminile.csv,15702
